In [1]:
suppressPackageStartupMessages({
  library(org.Hs.eg.db)
  library(AnnotationDbi)
  library(dplyr)
})

set.seed(20260521)

outdir <- "/data/work/rnaseq_matrix_demo"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

outdir

[1] "/data/work/rnaseq_matrix_demo"

In [2]:
all_symbols <- keys(org.Hs.eg.db, keytype = "SYMBOL")

# 保留格式比较规范的 gene symbol
all_symbols <- all_symbols[
  grepl("^[A-Za-z0-9.-]+$", all_symbols)
]

# 去重
all_symbols <- unique(all_symbols)

length(all_symbols)
head(all_symbols)

[1] 193167

[1] "A1BG"  "A2M"   "A2MP1" "NAT1"  "NAT2"  "NATP"

In [3]:
up_genes <- c(
  "IL1B", "IL6", "TNF", "CXCL8", "CXCL10", "CCL2", "CCL3", "CCL4",
  "NFKB1", "NFKBIA", "RELA", "STAT1", "STAT3", "IRF1", "IRF7",
  "ISG15", "IFI6", "IFI44", "IFIT1", "IFIT2", "IFIT3", "MX1",
  "OAS1", "OAS2", "HLA-A", "HLA-B", "HLA-C", "HLA-DRA",
  "HLA-DRB1", "CD74", "S100A8", "S100A9", "LYZ", "LST1",
  "HMOX1", "VEGFA"
)

down_genes <- c(
  "CD3D", "CD3E", "CD4", "CD8A", "CD8B",
  "MS4A1", "CD79A", "CD79B",
  "NKG7", "GNLY", "GZMB", "PRF1",
  "MKI67", "TOP2A", "PCNA", "MCM2", "MCM3", "MCM4",
  "MCM5", "MCM6", "CDK1", "CCNB1", "CCNB2", "AURKA", "AURKB"
)

# 确保这些基因都在 org.Hs.eg.db 中
up_genes <- intersect(up_genes, all_symbols)
down_genes <- intersect(down_genes, all_symbols)

length(up_genes)
length(down_genes)

[1] 36

[1] 25

In [4]:
n_total_genes <- 3000

de_genes <- unique(c(up_genes, down_genes))

background_genes <- setdiff(all_symbols, de_genes)
background_genes <- sample(background_genes, n_total_genes - length(de_genes))

genes <- unique(c(de_genes, background_genes))

length(genes)
head(genes)

[1] 3000

[1] "IL1B"   "IL6"    "TNF"    "CXCL8"  "CXCL10" "CCL2"

In [5]:
sample_info <- data.frame(
  sample_id = c(paste0("control_", 1:3), paste0("disease_", 1:3)),
  group = c(rep("control", 3), rep("disease", 3)),
  stringsAsFactors = FALSE
)

rownames(sample_info) <- sample_info$sample_id

sample_info

,sample_id,group
,<chr>,<chr>
control_1,control_1,control
control_2,control_2,control
control_3,control_3,control
disease_1,disease_1,disease
disease_2,disease_2,disease
disease_3,disease_3,disease


In [6]:
n_genes <- length(genes)
n_samples <- nrow(sample_info)

# 基因基础表达量：log-normal 分布，模拟不同基因表达量跨度
base_mean <- rlnorm(n_genes, meanlog = log(100), sdlog = 1.2)
names(base_mean) <- genes
base_mean[base_mean < 1] <- 1

# 样本测序深度差异
lib_size_factor <- c(0.95, 1.05, 1.00, 1.10, 0.90, 1.05)
names(lib_size_factor) <- sample_info$sample_id

# 设置 disease/control 的真实 fold change
fold_change <- rep(1, n_genes)
names(fold_change) <- genes

fold_change[up_genes] <- 4       # disease 上调，约 log2FC = 2
fold_change[down_genes] <- 0.25  # disease 下调，约 log2FC = -2

# Negative binomial dispersion
dispersion <- 0.15
size <- 1 / dispersion

count_mat <- matrix(
  0,
  nrow = n_genes,
  ncol = n_samples,
  dimnames = list(genes, sample_info$sample_id)
)

for (gene in genes) {
  for (sample in sample_info$sample_id) {
    group <- sample_info[sample, "group"]
    
    mu <- base_mean[gene] * lib_size_factor[sample]
    
    if (group == "disease") {
      mu <- mu * fold_change[gene]
    }
    
    count_mat[gene, sample] <- rnbinom(1, mu = mu, size = size)
  }
}

# 过滤极低表达基因
count_mat <- count_mat[rowSums(count_mat) >= 10, ]

dim(count_mat)
head(count_mat)

[1] 2997    6

,control_1,control_2,control_3,disease_1,disease_2,disease_3
IL1B,77,27,37,206,188,174
IL6,80,327,525,1704,1279,1948
TNF,19,19,20,97,54,103
CXCL8,14,9,8,31,37,44
CXCL10,91,47,67,359,442,556
CCL2,74,93,49,202,308,350


In [7]:
truth_de <- data.frame(
  gene = c(up_genes, down_genes),
  true_direction = c(
    rep("up_in_disease", length(up_genes)),
    rep("down_in_disease", length(down_genes))
  ),
  stringsAsFactors = FALSE
)

write.table(
  count_mat,
  file = file.path(outdir, "count_matrix.tsv"),
  sep = "\t",
  quote = FALSE,
  col.names = NA
)

write.table(
  sample_info,
  file = file.path(outdir, "sample_info.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

write.table(
  truth_de,
  file = file.path(outdir, "truth_DE_genes.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

list.files(outdir)

[1] "00_simdata.ipynb"           "01_RNA-seq_annotated.ipynb"
[3] "count_matrix.tsv"           "sample_info.tsv"           
[5] "teaching_figures"           "teaching_results"          
[7] "truth_DE_genes.tsv"